# Does Size Matter -- Training Size Experiment

Ed mentions that OpenAI recommends 50-100 examples for fine-tuning, but what actually happens when you vary the training set size?

This notebook fine-tunes GPT-4.1-nano with 50, 100, 200, 400, 1000, 2000, and 5000 training examples, then plots how accuracy changes. All 7 jobs run in parallel on OpenAI, same wait as one job.

**Dataset:** `ed-donner/items_lite` (20K Amazon products with summaries + prices, $0.50-$999)

In [ ]:
# imports

import os
import re
import io
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from datasets import load_dataset
from openai import OpenAI
from tqdm.notebook import tqdm

In [ ]:
# setup

load_dotenv(override=True)
client = OpenAI()

ds = load_dataset("ed-donner/items_lite")
train = ds["train"]
test = ds["test"]

print(f"Train: {len(train):,} items | Test: {len(test):,} items")
print(f"\nSample item:")
print(f"  Title: {train[0]['title']}")
print(f"  Price: ${train[0]['price']:.2f}")
print(f"  Summary: {train[0]['summary'][:200]}...")

## Evaluation helpers

In [ ]:
# post-processing: extract a numeric price from LLM output

def post_process(value):
    if isinstance(value, str):
        value = value.replace("$", "").replace(",", "")
        match = re.search(r"[-+]?\d*\.\d+|\d+", value)
        if not match:
            return 0
        price = float(match.group())
    else:
        price = float(value)
    
    # Clip to realistic range
    price = max(1.0, min(999.0, price))
    
    # Round to nearest common retail ending (.99, .95, .00)
    endings = [0.00, 0.49, 0.95, 0.99]
    base = int(price)
    best = min(endings, key=lambda e: abs(price - (base + e)))
    return round(base + best, 2)

In [ ]:
GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"

def color_for(error, truth):
    if error < 40 or error / truth < 0.2:
        return GREEN
    elif error < 80 or error / truth < 0.4:
        return YELLOW
    return RED


def evaluate(predictor, dataset, n=200):
    """Run predictor on n test items. Returns (predictions, actuals, mae)."""
    preds, actuals = [], []
    for i in tqdm(range(n)):
        item = dataset[i]
        guess = post_process(predictor(item))
        truth = item["price"]
        error = abs(guess - truth)
        c = color_for(error, truth)
        print(f"{c}${error:.0f}{RESET} ", end="")
        preds.append(guess)
        actuals.append(truth)
    mae = np.mean(np.abs(np.array(preds) - np.array(actuals)))
    print(f"\n\nMAE: ${mae:.2f}")
    return preds, actuals, mae

## Baselines

In [ ]:
# random baseline: guess a random price between $1 and $999

random.seed(42)

def random_predictor(item):
    return random.randrange(1, 1000)

_, _, random_mae = evaluate(random_predictor, test)

In [ ]:
# mean baseline: always predict the training set average

train_avg = np.mean([item["price"] for item in train])
print(f"Training average price: ${train_avg:.2f}")

def mean_predictor(item):
    return train_avg

_, _, mean_mae = evaluate(mean_predictor, test)

## Zero-shot GPT-4.1-nano

This is our "0 training examples" data point for the learning curve.

In [ ]:
def zero_shot(item):
    prompt = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item['summary']}"
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10,
        temperature=0
    )
    return response.choices[0].message.content

zero_preds, zero_actuals, zero_mae = evaluate(zero_shot, test)

## Prepare and launch all 7 fine-tuning jobs

Training sizes: 50, 100, 200, 400, 1000, 2000, 5000. Same 50 validation examples for all.

In [ ]:
SIZES = [50, 100, 200, 400, 1000, 2000, 5000]

def messages_for(item):
    prompt = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item['summary']}"
    return [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": f"${item['price']:.2f}"}
    ]

def make_jsonl(dataset, n):
    lines = []
    for i in range(n):
        row = json.dumps({"messages": messages_for(dataset[i])})
        lines.append(row)
    return "\n".join(lines)

# build validation JSONL (same 50 items for all jobs)
val = ds["validation"]
val_jsonl = make_jsonl(val, 50)

print(f"Validation set: 50 items")
print(f"Sample line: {json.loads(val_jsonl.split(chr(10))[0])['messages'][0]['content'][:80]}...")

In [ ]:
# upload all files and launch all jobs (retries if rate-limited)

import time

jobs = {}

# upload shared validation file once
val_file = client.files.create(
    file=("val.jsonl", io.BytesIO(val_jsonl.encode())),
    purpose="fine-tune"
)
print(f"Uploaded validation file: {val_file.id}")

for size in SIZES:
    train_jsonl = make_jsonl(train, size)
    train_file = client.files.create(
        file=(f"train_{size}.jsonl", io.BytesIO(train_jsonl.encode())),
        purpose="fine-tune"
    )
    # retry loop: OpenAI allows max 6 concurrent jobs
    while True:
        try:
            job = client.fine_tuning.jobs.create(
                training_file=train_file.id,
                validation_file=val_file.id,
                model="gpt-4.1-nano-2025-04-14",
                seed=42,
                hyperparameters={"n_epochs": 1, "batch_size": 1},
                suffix=f"pricer-{size}"
            )
            break
        except Exception as e:
            if "rate" in str(e).lower():
                print(f"  Size {size}: rate-limited, waiting 60s for a slot...")
                time.sleep(60)
            else:
                raise
    jobs[size] = job.id
    print(f"  Size {size}: job {job.id} launched")

print(f"\nAll {len(jobs)} jobs launched.")

## Monitor jobs

Re-run this cell to check status. All should finish around the same time.

In [ ]:
# check status of all jobs (re-run until all show 'succeeded')

all_done = True
for size, job_id in jobs.items():
    status = client.fine_tuning.jobs.retrieve(job_id)
    model_name = status.fine_tuned_model or "(training...)"
    print(f"  Size {size}: {status.status} -> {model_name}")
    if status.status != "succeeded":
        all_done = False

if all_done:
    print("\nAll jobs complete!")
else:
    print("\nStill training... re-run this cell in a few minutes.")

## Evaluate all fine-tuned models

In [ ]:
# get the fine-tuned model names

ft_models = {}
for size, job_id in jobs.items():
    info = client.fine_tuning.jobs.retrieve(job_id)
    ft_models[size] = info.fine_tuned_model
    print(f"  Size {size}: {info.fine_tuned_model}")

In [ ]:
# evaluate each fine-tuned model on the same 200 test items

ft_results = {}

for size in SIZES:
    model_name = ft_models[size]
    print(f"\n--- Fine-tuned on {size} examples ({model_name}) ---")

    def ft_predictor(item, _model=model_name):
        prompt = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item['summary']}"
        response = client.chat.completions.create(
            model=_model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=10,
            temperature=0
        )
        return response.choices[0].message.content

    preds, actuals, mae = evaluate(ft_predictor, test)
    ft_results[size] = {"preds": preds, "actuals": actuals, "mae": mae}

## Results

In [ ]:
# summary table

print(f"{'Model':<25} {'MAE':>10}")
print("-" * 37)
print(f"{'Random baseline':<25} ${random_mae:>8.2f}")
print(f"{'Mean baseline':<25} ${mean_mae:>8.2f}")
print(f"{'Zero-shot GPT-4.1-nano':<25} ${zero_mae:>8.2f}")
for size in SIZES:
    print(f"{'Fine-tuned (' + str(size) + ' ex.)':<25} ${ft_results[size]['mae']:>8.2f}")

### Learning curve

In [ ]:
# learning curve: MAE vs training size

sizes_with_zero = [0] + SIZES
maes_curve = [zero_mae] + [ft_results[s]["mae"] for s in SIZES]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sizes_with_zero, maes_curve, "o-", color="#2563eb", linewidth=2, markersize=8)
ax.axhline(y=zero_mae, color="#f59e0b", linestyle="--", linewidth=1, label=f"Zero-shot baseline (${zero_mae:.0f})")

for x, y in zip(sizes_with_zero, maes_curve):
    offset = 12 if y >= zero_mae else -18
    ax.annotate(f"${y:.0f}", (x, y), textcoords="offset points",
                xytext=(0, offset), ha="center", fontsize=10)

ax.set_xlabel("Training examples")
ax.set_ylabel("Mean Absolute Error ($)")
ax.set_title("Learning Curve: How Much Training Data Does GPT-4.1-nano Need?")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### All approaches compared

In [ ]:
# bar chart: all approaches side-by-side

labels = ["Random", "Mean", "Zero-shot"] + [f"FT-{s}" for s in SIZES]
values = [random_mae, mean_mae, zero_mae] + [ft_results[s]["mae"] for s in SIZES]
colors = ["#94a3b8", "#94a3b8", "#f59e0b"] + [
    "#10b981" if ft_results[s]["mae"] < zero_mae else "#2563eb" for s in SIZES
]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(labels, values, color=colors)
ax.axhline(y=zero_mae, color="#f59e0b", linestyle="--", linewidth=1, alpha=0.7)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f"${val:.0f}", ha="center", fontsize=9)

ax.set_ylabel("Mean Absolute Error ($)")
ax.set_title("All Approaches Compared (green = beats zero-shot)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Predicted vs actual price

In [ ]:
# scatter plots: zero-shot vs best fine-tuned model

best_size = min(ft_results, key=lambda s: ft_results[s]["mae"])
best = ft_results[best_size]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

max_price = max(max(zero_actuals), max(zero_preds),
                max(best["actuals"]), max(best["preds"]))

for ax, preds, actuals, title in [
    (ax1, zero_preds, zero_actuals, f"Zero-shot (MAE=${zero_mae:.0f})"),
    (ax2, best["preds"], best["actuals"], f"Fine-tuned {best_size} ex. (MAE=${best['mae']:.0f})"),
]:
    ax.scatter(actuals, preds, alpha=0.4, s=15, color="#2563eb")
    ax.plot([0, max_price], [0, max_price], "--", color="#ef4444", linewidth=1)
    ax.set_xlabel("Actual price ($)")
    ax.set_ylabel("Predicted price ($)")
    ax.set_title(title)
    ax.set_xlim(0, max_price)
    ax.set_ylim(0, max_price)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()